# Exercices de la Rencontre 13 : IA (avancé), Synthèse et Révision

Ce notebook regroupe les exercices de la rencontre 13. Il comporte trois parties :

1. **IA avancé** : scikit-learn en pratique (régression sur de vraies données), un petit réseau de neurones et une estimation de l'empreinte écologique d'un entraînement ;
2. **Synthèse** : une analyse complète Pandas + NumPy + Matplotlib sur un nouveau jeu de données, et la correction d'erreurs fréquentes ;
3. **Révision** : des exercices de code à faire sans documentation, en préparation de l'examen formatif (R14) et de l'examen final (R15).

### Accès aux jeux de données

Exécutez la cellule suivante en premier. La fonction `chemin_donnees()` cherche d'abord le fichier CSV dans le même dossier que le notebook (comme demandé dans les exercices du cours). Si le fichier n'y est pas (par exemple dans Google Colab), elle utilise la copie en ligne du dépôt du cours.

In [ ]:
import os

URL_DONNEES = "https://raw.githubusercontent.com/Simonll/420-SN1/develop/web/static/ressources/csv/"


def chemin_donnees(nom_fichier):
    """Retourne le chemin local du fichier s'il existe, sinon son adresse en ligne."""
    for dossier in [".", "../../web/static/ressources/csv"]:
        chemin = os.path.join(dossier, nom_fichier)
        if os.path.exists(chemin):
            return chemin
    return URL_DONNEES + nom_fichier


print(chemin_donnees("tips_dataset.csv"))

## Partie 1 - IA avancé

### Exercice 1 - Prédire la consommation d'essence avec scikit-learn

Utilisez le fichier `mpg_dataset.csv`.

1. Chargez les données et créez une colonne `l_par_100km` = `235.215 / mpg` (conversion en litres aux 100 km) ;
2. Retirez les lignes avec des valeurs manquantes dans les colonnes utilisées (`l_par_100km`, `weight`, `cylinders`, `horsepower`) ;
3. Séparez les données en entraînement / test (`train_test_split`, `test_size=0.2`, `random_state=42`) ;
4. Entraînez un premier modèle `LinearRegression` avec `weight` seulement, puis un second avec `weight`, `cylinders` et `horsepower` ;
5. Comparez le R² des deux modèles **sur les données de test** ;
6. Prédisez la consommation d'une voiture de 3000 lb, 4 cylindres et 95 chevaux.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

df_mpg = pd.read_csv(chemin_donnees("mpg_dataset.csv"))
df_mpg["l_par_100km"] = 235.215 / df_mpg["mpg"]
df_mpg = df_mpg.dropna(subset=["l_par_100km", "weight", "cylinders", "horsepower"])

X = df_mpg[["weight", "cylinders", "horsepower"]]
y = df_mpg["l_par_100km"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

modele_simple = LinearRegression().fit(X_train[["weight"]], y_train)
modele_multiple = LinearRegression().fit(X_train, y_train)

print(f"R² test (weight seulement)          : {modele_simple.score(X_test[['weight']], y_test):.3f}")
print(f"R² test (weight, cylinders, horsep.) : {modele_multiple.score(X_test, y_test):.3f}")

nouvelle_voiture = pd.DataFrame({"weight": [3000], "cylinders": [4], "horsepower": [95]})
print(f"Consommation prédite : {modele_multiple.predict(nouvelle_voiture)[0]:.1f} L/100 km")

plt.scatter(X_test["weight"], y_test, alpha=0.6, label="Données de test")
x_poids = pd.DataFrame({"weight": np.linspace(X["weight"].min(), X["weight"].max(), 100)})
plt.plot(x_poids["weight"], modele_simple.predict(x_poids), color="crimson", label="Modèle simple")
plt.title("Consommation en fonction du poids")
plt.xlabel("Poids (lb)")
plt.ylabel("Consommation (L/100 km)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Exercice 2 - Un petit réseau de neurones (deep learning, vue d'ensemble)

Le *deep learning* repose sur des réseaux de neurones à plusieurs couches. Les notes de cours présentent TensorFlow/Keras ; ici, pour garder un notebook léger, on utilise le réseau de neurones de scikit-learn (`MLPClassifier`), qui suit le même principe.

Le jeu de données `load_digits` (inclus dans scikit-learn) contient 1797 images de chiffres manuscrits de 8 x 8 pixels.

1. Chargez les données et affichez la forme de `X` et de `y` ;
2. Affichez les 5 premières images avec `plt.imshow` ;
3. Séparez en entraînement / test (`test_size=0.25`, `random_state=0`) ;
4. Entraînez un `MLPClassifier` avec une couche cachée de 64 neurones (`hidden_layer_sizes=(64,)`, `max_iter=500`, `random_state=0`) ;
5. Mesurez le temps d'entraînement et affichez la précision sur les données de test.

In [ ]:
import time
from sklearn.datasets import load_digits
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

chiffres = load_digits()
X_img, y_img = chiffres.data, chiffres.target
print("Forme de X :", X_img.shape, "| forme de y :", y_img.shape)

fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for i in range(5):
    axes[i].imshow(chiffres.images[i], cmap="gray_r")
    axes[i].set_title(f"Étiquette : {y_img[i]}")
    axes[i].axis("off")
plt.show()

X_train, X_test, y_train, y_test = train_test_split(X_img, y_img, test_size=0.25, random_state=0)

reseau = MLPClassifier(hidden_layer_sizes=(64,), max_iter=500, random_state=0)
debut = time.perf_counter()
reseau.fit(X_train, y_train)
duree_entrainement = time.perf_counter() - debut

precision = accuracy_score(y_test, reseau.predict(X_test))
print(f"Temps d'entraînement : {duree_entrainement:.2f} s")
print(f"Précision sur le test : {precision * 100:.1f} %")

### Exercice 3 - Estimer l'empreinte écologique d'un entraînement

L'énergie consommée par un entraînement se calcule avec : **énergie (kWh) = puissance (kW) x durée (h)**. Les émissions de CO₂ équivalent dépendent ensuite de la source d'électricité : **émissions (kg CO₂e) = énergie (kWh) x facteur d'émission (kg CO₂e/kWh)**.

Pour cet exercice, utilisez les valeurs approximatives suivantes (ordres de grandeur seulement) :

| Hypothèse | Valeur |
|-----------|--------|
| Puissance d'un ordinateur portable | 50 W |
| Puissance d'un serveur de calcul avec 8 GPU | 6000 W |
| Facteur d'émission, électricité au Québec (hydroélectricité) | 0,002 kg CO₂e/kWh |
| Facteur d'émission, moyenne mondiale | 0,5 kg CO₂e/kWh |

1. Écrivez une fonction `emissions_kg(puissance_w, duree_h, facteur)` qui retourne les émissions en kg CO₂e ;
2. Estimez les émissions de l'entraînement de l'exercice 2 sur un ordinateur portable (utilisez `duree_entrainement`) ;
3. Estimez les émissions d'un entraînement de 30 jours sur le serveur, au Québec et avec la moyenne mondiale ;
4. Combien de fois plus d'émissions pour le serveur (moyenne mondiale) que pour votre petit réseau (Québec) ?

In [ ]:
FACTEUR_QUEBEC = 0.002   # kg CO2e / kWh (approximation)
FACTEUR_MONDE = 0.5      # kg CO2e / kWh (approximation)


def emissions_kg(puissance_w, duree_h, facteur):
    energie_kwh = puissance_w / 1000 * duree_h
    return energie_kwh * facteur


petit_reseau = emissions_kg(50, duree_entrainement / 3600, FACTEUR_QUEBEC)
serveur_quebec = emissions_kg(6000, 30 * 24, FACTEUR_QUEBEC)
serveur_monde = emissions_kg(6000, 30 * 24, FACTEUR_MONDE)

print(f"Petit réseau (portable, Québec) : {petit_reseau * 1000:.6f} g CO2e")
print(f"Serveur 30 jours (Québec)       : {serveur_quebec:.1f} kg CO2e")
print(f"Serveur 30 jours (monde)        : {serveur_monde:.1f} kg CO2e")
print(f"Rapport serveur (monde) / petit réseau (Québec) : {serveur_monde / petit_reseau:.1e}")

# À retenir : la taille du modèle, la durée d'entraînement et la source d'électricité
# changent l'empreinte de plusieurs ordres de grandeur. Choisir un modèle sobre,
# adapté au problème, est une bonne pratique dès maintenant.

## Partie 2 - Synthèse

### Exercice 4 - Synthèse complète avec un nouveau jeu de données

Utilisez le fichier `Turing.csv` et réalisez l'analyse complète :

1. **Exploration** : charger, `info()`, `dtypes`, `describe()`, `shape` ;
2. **Filtrage** : lauréats affiliés à MIT ou à Stanford, lauréats entre 1980 et 1990 ;
3. **Création de colonne** : décennie à partir de l'année ;
4. **Agrégation** : nombre de lauréats par décennie et par affiliation ;
5. **NumPy** : extraire la colonne `year` en tableau NumPy, calculer des statistiques, créer un masque ;
6. **Visualisation** (Matplotlib, sans les graphiques de Pandas) : histogramme des années, diagramme à barres des lauréats par décennie ;
7. **Export** : exporter le DataFrame filtré dans un fichier CSV.

In [ ]:
# 1. Exploration
df_turing = pd.read_csv(chemin_donnees("Turing.csv"))
print("Forme :", df_turing.shape)
print(df_turing.dtypes)
df_turing.info()
print(df_turing.describe())

In [ ]:
# 2. Filtrage
masque_mit_stanford = df_turing["affiliation"].isin(["MIT", "Stanford University"])
df_mit_stanford = df_turing[masque_mit_stanford]
print(f"Lauréats MIT ou Stanford : {len(df_mit_stanford)}")
print(df_mit_stanford[["year", "laureate", "affiliation"]].head())

masque_80s = (df_turing["year"] >= 1980) & (df_turing["year"] <= 1990)
print(f"\nLauréats de 1980 à 1990 : {masque_80s.sum()}")

# 3. Création de colonne
df_turing["decennie"] = (df_turing["year"] // 10) * 10

# 4. Agrégation
par_decennie = df_turing.groupby("decennie")["laureate"].count()
print("\nLauréats par décennie :")
print(par_decennie)
print("\nAffiliations les plus fréquentes :")
print(df_turing.groupby("affiliation")["laureate"].count().sort_values(ascending=False).head(5))

In [ ]:
# 5. NumPy
annees = df_turing["year"].to_numpy()
print(f"Année moyenne : {annees.mean():.1f}")
print(f"Médiane       : {np.median(annees):.0f}")
print(f"Première / dernière : {annees.min()} / {annees.max()}")
masque_2000 = annees >= 2000
print(f"Prix remis depuis 2000 : {masque_2000.sum()} sur {annees.size}")

# 6. Visualisation
plt.hist(annees, bins=12, color="steelblue", edgecolor="white")
plt.title("Années de remise du prix Turing")
plt.xlabel("Année")
plt.ylabel("Nombre de lauréats")
plt.savefig("turing_histogramme.png")
plt.show()

etiquettes = [f"{d}s" for d in par_decennie.index]
plt.bar(etiquettes, par_decennie.to_numpy(), color="seagreen")
plt.title("Lauréats du prix Turing par décennie")
plt.xlabel("Décennie")
plt.ylabel("Nombre de lauréats")
plt.savefig("turing_decennies.png")
plt.show()

# 7. Export
df_mit_stanford.to_csv("turing_mit_stanford.csv", index=False)
print("Lignes relues depuis l'export :", len(pd.read_csv("turing_mit_stanford.csv")))

### Exercice 5 - Correction des erreurs fréquentes

Chacun des extraits suivants contient une erreur classique. Pour chacun : lisez le code fautif (en commentaire), expliquez l'erreur, puis vérifiez que la correction fonctionne.

**Erreur 1 :** modifier un tuple.

```python
coordonnees = (45.5, -73.5)
coordonnees[0] = 46.0  # TypeError : un tuple est immuable
```

In [ ]:
coordonnees = (45.5, -73.5)
# Correction : créer un nouveau tuple
coordonnees = (46.0, coordonnees[1])
print(coordonnees)

**Erreur 2 :** modifier un ensemble pendant qu'on le parcourt.

```python
ensemble = {1, 2, 3, 4}
for element in ensemble:
    if element % 2 == 0:
        ensemble.remove(element)  # RuntimeError : la taille change pendant l'itération
```

In [ ]:
ensemble = {1, 2, 3, 4}
# Correction : construire d'abord l'ensemble des éléments à retirer
a_supprimer = {element for element in ensemble if element % 2 == 0}
ensemble = ensemble - a_supprimer
print(ensemble)

**Erreur 3 :** confondre `loc` (étiquettes) et `iloc` (positions).

```python
df = pd.DataFrame({'A': [1, 2, 3], 'B': [4, 5, 6]})
df.loc[0, 0]  # KeyError : il n'y a pas de colonne nommée 0
```

In [ ]:
df = pd.DataFrame({"A": [1, 2, 3], "B": [4, 5, 6]})
print("loc  (étiquettes) :", df.loc[0, "A"])
print("iloc (positions)  :", df.iloc[0, 0])

## Partie 3 - Révision (préparation des examens)

Faites les exercices suivants **sans documentation** et sans aide extérieure, comme lors de l'examen. N'utilisez pas les fonctions toutes faites indiquées entre parenthèses.

### Exercice 6 - Code à écrire sans documentation

1. Écrivez une fonction `renverser(liste)` qui renverse une liste (sans `reverse()` ni `[::-1]`) ;
2. Écrivez une fonction `compter(liste, element)` qui compte les occurrences d'un élément (sans `count()`) ;
3. Écrivez une fonction `maximum(liste)` qui trouve le maximum d'une liste (sans `max()`) ;
4. Écrivez une fonction `garder_pairs(liste)` qui ne garde que les éléments pairs ;
5. Écrivez une fonction `moyenne(liste)` qui calcule la moyenne d'une liste (sans `sum()`).

In [ ]:
def renverser(liste):
    resultat = []
    for i in range(len(liste) - 1, -1, -1):
        resultat.append(liste[i])
    return resultat


def compter(liste, element):
    total = 0
    for valeur in liste:
        if valeur == element:
            total += 1
    return total


def maximum(liste):
    plus_grand = liste[0]
    for valeur in liste:
        if valeur > plus_grand:
            plus_grand = valeur
    return plus_grand


def garder_pairs(liste):
    return [valeur for valeur in liste if valeur % 2 == 0]


def moyenne(liste):
    total = 0
    for valeur in liste:
        total += valeur
    return total / len(liste)


donnees = [4, 7, 1, 7, 10, 3, 7, 8]
print("renverser    :", renverser(donnees))
print("compter 7    :", compter(donnees, 7))
print("maximum      :", maximum(donnees))
print("garder_pairs :", garder_pairs(donnees))
print("moyenne      :", moyenne(donnees))

### Exercice 7 - Prédire l'affichage

Sur papier, écrivez ce que le programme suivant affichera. Exécutez ensuite la cellule pour vérifier votre réponse.

In [ ]:
import numpy as np

valeurs = np.array([[3, 8, 1], [6, 2, 9]])
total = 0
for i in range(valeurs.shape[0]):
    for j in range(valeurs.shape[1]):
        if valeurs[i, j] % 2 == 0:
            total += valeurs[i, j]
            print(i, j, total)
print("Total :", total)
print(valeurs[1, :2], valeurs[:, -1])

## Poursuite du TP2

C'est la dernière rencontre de travail avant la **remise du TP2 en R14**. Profitez-en pour finaliser votre code et vos visualisations, rédiger votre compte-rendu et vérifier que tout est poussé sur GitHub.

En petit groupe, prenez aussi quelques minutes pour discuter : quelles notions de R9 à R12 avez-vous trouvées les plus difficiles ? Quelles stratégies de révision prévoyez-vous pour l'examen formatif ?